In [2]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score
from tensorflow.keras.callbacks import EarlyStopping

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
data = pd.read_parquet("train.parquet")


In [5]:
data.columns

Index(['seq_ix', 'step_in_seq', 'need_prediction', '0', '1', '2', '3', '4',
       '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17',
       '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29',
       '30', '31'],
      dtype='object')

In [6]:
arr = data.drop(columns=["step_in_seq","need_prediction","seq_ix"]).values

In [7]:
arr = arr.reshape(517,1000,32)

In [8]:
arr.shape

(517, 1000, 32)

In [43]:
window_size = 49
num_samples = 1000//(window_size+1)
X_train = np.zeros(shape=(517*num_samples,49,32),dtype=np.float32)
y_train = np.zeros(shape=(517*num_samples,32),dtype=np.float32)
for i in range(X_train.shape[0]):
    j = i% 517
    k = i//517
    X_train[i] = arr[k][j:j+window_size][:]
    y_train[i] = arr[k][j+window_size][:]


In [44]:
X_train.shape
y_train.shape

(10340, 32)

In [51]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [45]:
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.layers import Softmax, Multiply ,Lambda

In [59]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Softmax, Multiply, Lambda

def build_model():
   
    neurons = [32, 28,42]
    window_size = 49
    num_features = 32

    
    inputs = Input(shape=(window_size, num_features))
    enc1_out = LSTM(
        neurons[0], 
        return_sequences=True, 
        name="encoder1", 
        recurrent_dropout=0.3
    )(inputs)
    
    enc2_out = LSTM(
        neurons[1], 
        return_sequences=True, 
        name="encoder2"
    )(enc1_out)

    
    attention_scores = Dense(64, activation='tanh')(enc2_out)
    attention_scores = Dense(1, activation="linear")(attention_scores)
    
    attention_weights = Softmax(axis=1)(attention_scores)
    
   
    weighted_sum = Multiply()([enc2_out, attention_weights])
    
    context_vector = Lambda(lambda x: tf.reduce_sum(x, axis=1))(weighted_sum)

    dense_2 = Dense(neurons[-1], name="dense_2", activation="tanh")(context_vector)
    drop = layers.Dropout(0.3)(dense_2)
    outputs = Dense(num_features,name="final", activation="linear")(drop)

    
    model = Model(inputs=inputs, outputs=outputs)
    return model




In [60]:
model = build_model()
optim = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optim, loss='mean_squared_error',metrics=[r2_keras])
model.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 49, 32)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder1 (LSTM)     │ (None, 49, 32)    │      8,320 │ input_layer_12[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder2 (LSTM)     │ (None, 49, 28)    │      6,832 │ encoder1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 49, 64)    │      1,856 │ encoder2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 49, 1)     │         65 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_12          │ (None, 49, 1)     │          0 │ dense_25[0][0]    │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_12         │ (None, 49, 28)    │          0 │ encoder2[0][0],   │
│ (Multiply)          │                   │            │ softmax_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_12 (Lambda)  │ (None, 28)        │          0 │ multiply_12[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 42)        │      1,218 │ lambda_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 42)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final (Dense)       │ (None, 32)        │      1,376 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,667 (76.82 KB)

 Trainable params: 19,667 (76.82 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:


print("Starting model training...")

# 🚀 Fit the model!
history = model.fit(
    X_train,   # Pass your training generator
    y_train,
    batch_size = 64,
    validation_split = 0.2,
    epochs=100,
)

print("🎉 Training complete!")

Starting model training...
Epoch 1/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 0.9296 - r2_keras: 0.0362 - val_loss: 0.7886 - val_r2_keras: 0.0315
Epoch 2/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.8287 - r2_keras: 0.1392 - val_loss: 0.7456 - val_r2_keras: 0.0825
Epoch 3/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.7745 - r2_keras: 0.1942 - val_loss: 0.7194 - val_r2_keras: 0.1134
Epoch 4/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.7460 - r2_keras: 0.2231 - val_loss: 0.7039 - val_r2_keras: 0.1316
Epoch 5/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.7294 - r2_keras: 0.2401 - val_loss: 0.6932 - val_r2_keras: 0.1440
Epoch 6/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.7181 - r2_keras: 0.2517 - val_loss: 0.6855 - val_r2_keras: 0.1529
Epoch 7/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.7140 - r2_keras: 0.2559 - val_loss: 0.6804 - val_r2_keras: 0.1589
Epoch 8/100
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0

KeyboardInterrupt: 